# Batch Record Exception Detection Agent
### Section 1 — Install & Import

NOTE: Real pharmaceutical batch records are proprietary and never publicly available.
This notebook generates SYNTHETIC batch records that follow real GMP structure
(based on publicly documented FDA guidance), with realistic exceptions injected.
In production, this would run on UCB's actual MES data behind GxP controls.

In [1]:
!pip install pandas numpy boto3 faker -q

import pandas as pd
import numpy as np
import json
import random
import os
import boto3
from datetime import datetime, timedelta
from faker import Faker

fake = Faker()
random.seed(42)
np.random.seed(42)

print("Done.")

Done.


---
### Section 2 — Define batch record structure

Based on publicly documented FDA GMP batch record requirements:
- Product identification
- Material and equipment used
- In-process test results
- Final release test results
- Sign-offs and approvals

In [2]:
# Products — using UCB's own AED portfolio
PRODUCTS = [
    {"name": "Levetiracetam Tablets",  "strength": "750mg", "spec_weight": (740, 760), "spec_hardness": (6, 10), "spec_dissolution": 80},
    {"name": "Brivaracetam Tablets",   "strength": "100mg", "spec_weight": (95, 105),  "spec_hardness": (5, 9),  "spec_dissolution": 85},
    {"name": "Lamotrigine Tablets",    "strength": "200mg", "spec_weight": (195, 205), "spec_hardness": (7, 11), "spec_dissolution": 80},
    {"name": "Topiramate Tablets",     "strength": "100mg", "spec_weight": (98, 102),  "spec_hardness": (6, 10), "spec_dissolution": 80},
]

EQUIPMENT = [
    {"id": "M-204", "name": "Tablet Mixer",      "calibration_interval_days": 90},
    {"id": "P-118", "name": "Tablet Press",      "calibration_interval_days": 60},
    {"id": "D-302", "name": "Dissolution Tester","calibration_interval_days": 30},
    {"id": "S-410", "name": "Scale",             "calibration_interval_days": 30},
]

OPERATORS = [fake.name() for _ in range(8)]
QA_REVIEWERS = [fake.name() for _ in range(4)]

print(f"Products tracked   : {len(PRODUCTS)}")
print(f"Equipment tracked  : {len(EQUIPMENT)}")
print(f"Operators          : {len(OPERATORS)}")
print(f"QA reviewers       : {len(QA_REVIEWERS)}")

Products tracked   : 4
Equipment tracked  : 4
Operators          : 8
QA reviewers       : 4


---
### Section 3 — Define exception types

These are the realistic exception categories a quality reviewer looks for
when reviewing a batch record before release.

In [3]:
EXCEPTION_TYPES = [
    {
        "code"       : "OOS_WEIGHT",
        "name"       : "Out-of-spec tablet weight",
        "severity"   : "High",
        "description": "Tablet weight falls outside the approved specification range.",
    },
    {
        "code"       : "OOS_HARDNESS",
        "name"       : "Out-of-spec hardness",
        "severity"   : "High",
        "description": "Tablet hardness falls outside the approved specification range.",
    },
    {
        "code"       : "OOS_DISSOLUTION",
        "name"       : "Out-of-spec dissolution",
        "severity"   : "Critical",
        "description": "Dissolution result is below the minimum required percentage.",
    },
    {
        "code"       : "MISSING_SIGNATURE",
        "name"       : "Missing QA signature",
        "severity"   : "Critical",
        "description": "Required sign-off is missing from the batch record.",
    },
    {
        "code"       : "EXPIRED_CALIBRATION",
        "name"       : "Equipment calibration expired",
        "severity"   : "Critical",
        "description": "Equipment used in manufacturing was outside its calibration window.",
    },
    {
        "code"       : "LOT_MISMATCH",
        "name"       : "Raw material lot mismatch",
        "severity"   : "High",
        "description": "Raw material lot number on the record does not match the issued lot.",
    },
    {
        "code"       : "TEMP_DEVIATION",
        "name"       : "Process temperature deviation",
        "severity"   : "Medium",
        "description": "Manufacturing process temperature deviated from the validated range.",
    },
    {
        "code"       : "TIMING_GAP",
        "name"       : "Unexplained timing gap",
        "severity"   : "Medium",
        "description": "Gap between process steps exceeds the validated maximum hold time.",
    },
]

print(f"Exception types defined: {len(EXCEPTION_TYPES)}")
for e in EXCEPTION_TYPES:
    print(f"  [{e['severity']:<8}] {e['code']:<22} {e['name']}")

Exception types defined: 8
  [High    ] OOS_WEIGHT             Out-of-spec tablet weight
  [High    ] OOS_HARDNESS           Out-of-spec hardness
  [Critical] OOS_DISSOLUTION        Out-of-spec dissolution
  [Critical] MISSING_SIGNATURE      Missing QA signature
  [Critical] EXPIRED_CALIBRATION    Equipment calibration expired
  [High    ] LOT_MISMATCH           Raw material lot mismatch
  [Medium  ] TEMP_DEVIATION         Process temperature deviation
  [Medium  ] TIMING_GAP             Unexplained timing gap


---
### Section 4 — Generate synthetic batch records

Each batch record includes:
- Product and batch identification
- Equipment used with calibration dates
- In-process test results
- Final release test results
- Operator and QA sign-offs

Approximately 20% of batches will have 1-2 injected exceptions — realistic
for a manufacturing environment with good quality controls.

In [4]:
def generate_batch_record(batch_num, inject_exception=False):
    """Generate one synthetic batch record, optionally with an injected exception."""

    product   = random.choice(PRODUCTS)
    equipment = random.sample(EQUIPMENT, 2)
    operator  = random.choice(OPERATORS)
    reviewer  = random.choice(QA_REVIEWERS)

    batch_date = fake.date_between(start_date="-180d", end_date="today")
    batch_id   = f"{product['name'][:2].upper()}-{batch_date.strftime('%Y%m')}-{batch_num:04d}"

    # Generate normal in-spec values
    weight_lo, weight_hi   = product["spec_weight"]
    hardness_lo, hardness_hi = product["spec_hardness"]

    weight     = round(np.random.uniform(weight_lo + 1, weight_hi - 1), 1)
    hardness   = round(np.random.uniform(hardness_lo + 0.3, hardness_hi - 0.3), 1)
    dissolution= round(np.random.uniform(product["spec_dissolution"] + 5, 99), 1)

    # Equipment calibration — normally valid
    cal_dates = {}
    for eq in equipment:
        last_cal = batch_date - timedelta(days=random.randint(1, eq["calibration_interval_days"] - 5))
        cal_dates[eq["id"]] = last_cal

    record = {
        "batch_id"           : batch_id,
        "product"            : f"{product['name']} {product['strength']}",
        "batch_date"         : str(batch_date),
        "operator"           : operator,
        "qa_reviewer"        : reviewer,
        "equipment_used"     : [eq["id"] for eq in equipment],
        "calibration_dates"  : {k: str(v) for k, v in cal_dates.items()},
        "tablet_weight_mg"   : weight,
        "weight_spec"        : f"{weight_lo}-{weight_hi}",
        "hardness_kpa"       : hardness,
        "hardness_spec"      : f"{hardness_lo}-{hardness_hi}",
        "dissolution_pct"    : dissolution,
        "dissolution_spec"   : f">={product['spec_dissolution']}",
        "qa_signature_present": True,
        "lot_number_match"   : True,
        "exceptions"         : [],
    }

    if inject_exception:
        exception_type = random.choice(EXCEPTION_TYPES)
        record["exceptions"].append(exception_type["code"])

        if exception_type["code"] == "OOS_WEIGHT":
            record["tablet_weight_mg"] = round(weight_hi + random.uniform(2, 8), 1)
        elif exception_type["code"] == "OOS_HARDNESS":
            record["hardness_kpa"] = round(hardness_lo - random.uniform(0.5, 2), 1)
        elif exception_type["code"] == "OOS_DISSOLUTION":
            record["dissolution_pct"] = round(product["spec_dissolution"] - random.uniform(3, 15), 1)
        elif exception_type["code"] == "MISSING_SIGNATURE":
            record["qa_signature_present"] = False
        elif exception_type["code"] == "EXPIRED_CALIBRATION":
            eq = random.choice(equipment)
            expired_date = batch_date - timedelta(days=eq["calibration_interval_days"] + random.randint(1, 10))
            record["calibration_dates"][eq["id"]] = str(expired_date)
        elif exception_type["code"] == "LOT_MISMATCH":
            record["lot_number_match"] = False
            record["lot_number_recorded"] = f"LOT-{random.randint(1000,9999)}"
            record["lot_number_issued"]   = f"LOT-{random.randint(1000,9999)}"
        elif exception_type["code"] == "TEMP_DEVIATION":
            record["process_temp_c"] = round(random.uniform(28, 35), 1)
            record["process_temp_spec"] = "20-25"
        elif exception_type["code"] == "TIMING_GAP":
            record["hold_time_hours"] = round(random.uniform(25, 48), 1)
            record["hold_time_spec_max"] = 24

    return record


# Generate 200 batch records — 20% with injected exceptions
print("Generating synthetic batch records...")
batch_records = []

for i in range(1, 201):
    inject = random.random() < 0.20
    record = generate_batch_record(i, inject_exception=inject)
    batch_records.append(record)

n_with_exceptions = sum(1 for r in batch_records if len(r["exceptions"]) > 0)
print(f"\nGenerated {len(batch_records)} batch records")
print(f"  With exceptions    : {n_with_exceptions}")
print(f"  Clean (no issues)  : {len(batch_records) - n_with_exceptions}")

Generating synthetic batch records...

Generated 200 batch records
  With exceptions    : 38
  Clean (no issues)  : 162


In [5]:
# Save batch records
with open("synthetic_batch_records.json", "w") as f:
    json.dump(batch_records, f, indent=2)

print("Saved to synthetic_batch_records.json")
print()
print("Sample record (clean):")
clean_sample = next(r for r in batch_records if len(r["exceptions"]) == 0)
print(json.dumps(clean_sample, indent=2))

Saved to synthetic_batch_records.json

Sample record (clean):
{
  "batch_id": "LE-202601-0001",
  "product": "Levetiracetam Tablets 750mg",
  "batch_date": "2026-01-08",
  "operator": "Patricia Barker",
  "qa_reviewer": "Kathleen Meyer",
  "equipment_used": [
    "D-302",
    "M-204"
  ],
  "calibration_dates": {
    "D-302": "2025-12-15",
    "M-204": "2025-12-25"
  },
  "tablet_weight_mg": 747.7,
  "weight_spec": "740-760",
  "hardness_kpa": 9.5,
  "hardness_spec": "6-10",
  "dissolution_pct": 95.2,
  "dissolution_spec": ">=80",
  "qa_signature_present": true,
  "lot_number_match": true,
  "exceptions": []
}


In [6]:
print("Sample record (with exception):")
exception_sample = next(r for r in batch_records if len(r["exceptions"]) > 0)
print(json.dumps(exception_sample, indent=2))

Sample record (with exception):
{
  "batch_id": "BR-202604-0013",
  "product": "Brivaracetam Tablets 100mg",
  "batch_date": "2026-04-19",
  "operator": "James Miller",
  "qa_reviewer": "Mary Gutierrez",
  "equipment_used": [
    "P-118",
    "D-302"
  ],
  "calibration_dates": {
    "P-118": "2026-03-12",
    "D-302": "2026-04-06"
  },
  "tablet_weight_mg": 98.4,
  "weight_spec": "95-105",
  "hardness_kpa": 5.6,
  "hardness_spec": "5-9",
  "dissolution_pct": 96.2,
  "dissolution_spec": ">=85",
  "qa_signature_present": true,
  "lot_number_match": false,
  "exceptions": [
    "LOT_MISMATCH"
  ],
  "lot_number_recorded": "LOT-4593",
  "lot_number_issued": "LOT-3266"
}


---
### Section 5 — Exploratory analysis

In [7]:
# Exception distribution
all_exceptions = [e for r in batch_records for e in r["exceptions"]]
exception_counts = pd.Series(all_exceptions).value_counts()

print("Exception type distribution:")
for code, count in exception_counts.items():
    exc_info = next(e for e in EXCEPTION_TYPES if e["code"] == code)
    print(f"  [{exc_info['severity']:<8}] {code:<22} {count} occurrences")

print(f"\nTotal exceptions injected: {len(all_exceptions)}")

Exception type distribution:
  [High    ] OOS_HARDNESS           10 occurrences
  [Critical] MISSING_SIGNATURE      6 occurrences
  [Medium  ] TEMP_DEVIATION         5 occurrences
  [Critical] EXPIRED_CALIBRATION    5 occurrences
  [High    ] LOT_MISMATCH           4 occurrences
  [High    ] OOS_WEIGHT             3 occurrences
  [Medium  ] TIMING_GAP             3 occurrences
  [Critical] OOS_DISSOLUTION        2 occurrences

Total exceptions injected: 38


In [8]:
# Product distribution
product_counts = pd.Series([r["product"] for r in batch_records]).value_counts()
print("Batches by product:")
print(product_counts)

Batches by product:
Brivaracetam Tablets 100mg     53
Lamotrigine Tablets 200mg      52
Levetiracetam Tablets 750mg    48
Topiramate Tablets 100mg       47
Name: count, dtype: int64


---
### Section 6 — Rule-based exception detection

Before using an LLM, run deterministic rule checks — this is what a real
quality system would do first. The LLM layer comes after, to explain
the findings in plain English.

In [9]:
def detect_exceptions_rule_based(record):
    """
    Deterministic rule checks against the batch record.
    Returns a list of detected exceptions with evidence.
    """
    findings = []

    # Weight check
    weight_lo, weight_hi = map(float, record["weight_spec"].split("-"))
    if not (weight_lo <= record["tablet_weight_mg"] <= weight_hi):
        findings.append({
            "code"    : "OOS_WEIGHT",
            "severity": "High",
            "evidence": f"Tablet weight {record['tablet_weight_mg']}mg is outside spec range {record['weight_spec']}mg",
        })

    # Hardness check
    hardness_lo, hardness_hi = map(float, record["hardness_spec"].split("-"))
    if not (hardness_lo <= record["hardness_kpa"] <= hardness_hi):
        findings.append({
            "code"    : "OOS_HARDNESS",
            "severity": "High",
            "evidence": f"Tablet hardness {record['hardness_kpa']}kPa is outside spec range {record['hardness_spec']}kPa",
        })

    # Dissolution check
    diss_min = float(record["dissolution_spec"].replace(">=", ""))
    if record["dissolution_pct"] < diss_min:
        findings.append({
            "code"    : "OOS_DISSOLUTION",
            "severity": "Critical",
            "evidence": f"Dissolution {record['dissolution_pct']}% is below minimum {diss_min}%",
        })

    # Signature check
    if not record["qa_signature_present"]:
        findings.append({
            "code"    : "MISSING_SIGNATURE",
            "severity": "Critical",
            "evidence": "QA signature is missing from the final release section",
        })

    # Lot number check
    if not record["lot_number_match"]:
        findings.append({
            "code"    : "LOT_MISMATCH",
            "severity": "High",
            "evidence": f"Recorded lot {record.get('lot_number_recorded','?')} does not match issued lot {record.get('lot_number_issued','?')}",
        })

    # Calibration check
    batch_date = datetime.strptime(record["batch_date"], "%Y-%m-%d").date()
    for eq_id, cal_date_str in record["calibration_dates"].items():
        cal_date = datetime.strptime(cal_date_str, "%Y-%m-%d").date()
        eq_info  = next(e for e in EQUIPMENT if e["id"] == eq_id)
        days_since_cal = (batch_date - cal_date).days
        if days_since_cal > eq_info["calibration_interval_days"]:
            findings.append({
                "code"    : "EXPIRED_CALIBRATION",
                "severity": "Critical",
                "evidence": f"Equipment {eq_id} calibrated {days_since_cal} days before batch run, exceeds {eq_info['calibration_interval_days']}-day interval",
            })

    # Temperature check
    if "process_temp_c" in record:
        if record["process_temp_c"] > 25 or record["process_temp_c"] < 20:
            findings.append({
                "code"    : "TEMP_DEVIATION",
                "severity": "Medium",
                "evidence": f"Process temperature {record['process_temp_c']}C outside validated range {record['process_temp_spec']}C",
            })

    # Hold time check
    if "hold_time_hours" in record:
        if record["hold_time_hours"] > record["hold_time_spec_max"]:
            findings.append({
                "code"    : "TIMING_GAP",
                "severity": "Medium",
                "evidence": f"Hold time {record['hold_time_hours']}h exceeds validated maximum {record['hold_time_spec_max']}h",
            })

    return findings


# Run rule-based detection on all batch records
print("Running rule-based detection on all batch records...")
detection_log = []

for record in batch_records:
    findings = detect_exceptions_rule_based(record)
    detection_log.append({
        "batch_id"        : record["batch_id"],
        "product"         : record["product"],
        "true_exceptions" : record["exceptions"],
        "detected"        : findings,
        "n_true"          : len(record["exceptions"]),
        "n_detected"      : len(findings),
    })

# Quick accuracy check
correct = sum(1 for d in detection_log if set(d["true_exceptions"]) == set(f["code"] for f in d["detected"]))
print(f"\nExact match accuracy: {correct}/{len(detection_log)} ({correct/len(detection_log)*100:.1f}%)")

Running rule-based detection on all batch records...

Exact match accuracy: 200/200 (100.0%)


---
### Section 7 — Generate plain English review summary via Bedrock

For each batch with detected exceptions, generate a quality reviewer-friendly
summary explaining what was found and what action is needed.

In [ ]:
os.environ["AWS_BEARER_TOKEN_BEDROCK"] = "bedrock-api-key-YmVkcm9jay5hbWF6b25hd3MuY29tLz9BY3Rpb249Q2FsbFdpdGhCZWFyZXJUb2tlbiZYLUFtei1BbGdvcml0aG09QVdTNC1ITUFDLVNIQTI1NiZYLUFtei1DcmVkZW50aWFsPUFTSUE0WDZWRFZOWU5NSEE2SjM0JTJGMjAyNjA2MzAlMkZhcC1zb3V0aGVhc3QtMiUyRmJlZHJvY2slMkZhd3M0X3JlcXVlc3QmWC1BbXotRGF0ZT0yMDI2MDYzMFQwNTQzMTVaJlgtQW16LUV4cGlyZXM9NDMyMDAmWC1BbXotU2VjdXJpdHktVG9rZW49SVFvSmIzSnBaMmx1WDJWakVQYiUyRiUyRiUyRiUyRiUyRiUyRiUyRiUyRiUyRiUyRndFYURtRndMWE52ZFhSb1pXRnpkQzB5SWtnd1JnSWhBSWZVcjFTSzNkTWEyNFZLNTMlMkJaNngzMXFTU3ZRQ2xUOFNNdzVBYiUyRnZ5SEFBaUVBNUViMjM3dWhOS1E5aG1vdWUlMkJ4OTVYR3Q5VkV0YWxkc01QTlJOMDlyZ2RvcWtBVUl2JTJGJTJGJTJGJTJGJTJGJTJGJTJGJTJGJTJGJTJGJTJGQVJBQUdndzROell3T0RNek9URXpORFFpRExYOUwycDRwQjhIQ1M0RlppcmtCQUhCc09CU1VQcm1pUzM2VnVHM3p2YzJaZERpVVZCT0dsWmUzZXVaM3FYT2Rvc1pBdGtZQmw2JTJGV1RaSGJ3eGJDQ2M4eWs2bHptMDZ6cmdDRnU4NGM1R0RuOFhnYUx0QllhMHVHZEpyaU9USldER2FUTFdIQ055S3JmMUR3NllUNVhvSmRoVTRsUFNOJTJCcVhPM2tDcmZLeXlQYXJvTGtXemtHZHVkN2FtaVNBN2RIMGp5NTNGS0toeW9aRHhoemFIdGNHWm42SUZmRldWeU5yQ2dSY2FOU3p5ZzFJSmV4eFdLSzdGREFDbE1MMGNvUzRaOFh2UUxhWjZ2OGNWYm5UWCUyRlo3UGE2Y3BqaVlIVEtSMjNQd2xIbjNSREozaGNaMDcyV2QlMkZOeFZ1Rlc5OE1uWmdEU1Nkc1NTdDJReUVNUHZVNDByaE56RkNxSUR4Zkw5c25JazNzcU1FM1JOc2FwTm1CcVVYd1BkSXdDNGpsTWJDam1aOTNMR0ZaRkMyT3Nra2M0cUhxVXVXVEx4YkkxUmtXSCUyRklobWlZem93WFNZd2NURjdqZXBnM0Y1bTFPYmlsQW9HNUF5cFd3V2J0QWp1QVVmYWZMdTlhMzRPV29WN0VYcHNSTkRKQlZpNVl5OXNIR3VrckVYOGVFOGtQcU91Szg1RXNVN01pekZLU3hLOTYyVWNzaUg2SjIlMkZKOTdhSFQlMkJCJTJCQk5vcE1WWUNnWGElMkJRRWtqRXRFOFhtc0F0clZySGRJRjlWSFVmVUZNUm96YkVycmRYRUs3S05mdEZNU0pLN1M3UlNDSkMzS2hzYUZCdlRnVVB0eSUyQnlZcDk4dEFqMjhvN0xKWWklMkJPUmh2Y2MyNVU5QnN0T1JVV3cxY2FSMlh4QzB0OEFHYThBQUZTYnJ5UVVkY0J1U0xqT2NwR2x0bFloMXhycGtmVldkajIyaG1OZnYwWGw3eFY2OWJpNnolMkJDQ1VGRmRKRmJwNWhnSldPaUw2UjFNaHJITmFKYmFxNFl4U3cwU2x4NzdJM09yeGxpMmc1OUhmQ0IlMkJvajNJNDNxeENFSWJxVDRScUx3blJKQ20lMkZkOXd6NjNpeWlXNGdkUlREZnJvM1NCanJHQXM4a25nRE1TQkgyQ2c5anA1U2NoN05kdFFhMmVqWnVrcENVOGNONmN1aVRwT21vUm9rQWt5UUhUTk0lMkJyQjNZOXVJaGZENzVJOFQwYkFBSldxUFcwellyZFZpVVFKeEFTejhYUFdpTDVoN1d4Y2lISjZZcmRkSTVsOEw1ZDQ5U2FLMjQ5ckNrTExxRUFIZW1nMndleXdSZ0ZDRTZNOXNtVzV6eWlnWDBiSyUyRkdad1hXWTZucElONUJQTCUyQlYyU1BFVGZlUWpQUWNaOFZVR2QlMkZTQnJtODNMb05oTHNoek1oZzJkYnZFWDdUZlhReUZlJTJGYzlNSGU0N1pqVkhNTGJiZnk3MkkyZ2dFVyUyRlclMkIlMkZoWG8lMkJBYmZWd2lUMmV1R2VvQkNqQktvSTVLRjFKeWQxWUhKcmFyMnNjMmJsMHF6dEtUJTJCOWEzU3Y2OG4lMkZFeHRpWENvR0ZBREtIVWIlMkJUczVhYUtVQzFMMCUyQlZCQkozaEREMEhjc1B4OUFhS3liNE0xTnV5RndNWldwTFVIeHllWmlNRm0xdjhZWkNNWGNPdE9UZTVKQ3RtJTJCSHN1NyUyRmVDY1JsdlBYZDJwWiZYLUFtei1TaWduYXR1cmU9NmUxNjRiNDZhYmYyN2EwMWFiNjJmNDY3NDU1YmJkMmRjNDQ0OWE2NTEwOWMxMjgwMzAxNDMxNGFmNmFkMDZhOSZYLUFtei1TaWduZWRIZWFkZXJzPWhvc3QmVmVyc2lvbj0x"

bedrock   = boto3.client("bedrock-runtime", region_name="ap-southeast-2")
LLM_MODEL = "amazon.nova-pro-v1:0"

SYSTEM_PROMPT = """You are a pharmaceutical quality assurance assistant.
Review batch record exception findings and generate a clear summary for the
QA reviewer who will make the final batch disposition decision.
For each exception include: what was found, why it matters, and the
recommended next step per standard GMP practice.
Be direct and professional. Maximum 200 words."""


def generate_review_summary(batch_id, product, findings):
    """Generate a plain English batch review summary via Bedrock."""

    findings_text = "\n".join([
        f"- [{f['severity']}] {f['code']}: {f['evidence']}"
        for f in findings
    ])

    prompt = f"""Batch record review — {batch_id}
Product: {product}

Findings from automated review:
{findings_text}

Generate a QA review summary with recommended disposition action."""

    response = bedrock.converse(
        modelId=LLM_MODEL,
        system=[{"text": SYSTEM_PROMPT}],
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": 350, "temperature": 0}
    )
    return response["output"]["message"]["content"][0]["text"]


print("Bedrock client ready.")

Bedrock client ready.


In [ ]:
# Generate review summaries for all batches with exceptions
print("Generating QA review summaries for flagged batches...\n")
print("=" * 70)

review_summaries = []

flagged_batches = [d for d in detection_log if len(d["detected"]) > 0]

for d in flagged_batches:
    summary = generate_review_summary(d["batch_id"], d["product"], d["detected"])

    review_summaries.append({
        "batch_id"  : d["batch_id"],
        "product"   : d["product"],
        "n_findings": len(d["detected"]),
        "findings"  : d["detected"],
        "summary"   : summary,
    })

    print(f"BATCH REVIEW — {d['batch_id']} ({d['product']})")
    print(f"Findings: {len(d['detected'])}")
    print(f"\n{summary}")
    print("=" * 70)

print(f"\nTotal review summaries generated: {len(review_summaries)}")

---
### Section 8 — Evaluate detection performance

In [ ]:
# Evaluate rule-based detection against ground truth
tp, fp, fn = 0, 0, 0

for d in detection_log:
    true_set     = set(d["true_exceptions"])
    detected_set = set(f["code"] for f in d["detected"])

    tp += len(true_set & detected_set)
    fp += len(detected_set - true_set)
    fn += len(true_set - detected_set)

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("DETECTION PERFORMANCE")
print("=" * 40)
print(f"  True Positives  : {tp}")
print(f"  False Positives : {fp}")
print(f"  False Negatives : {fn}")
print()
print(f"  Precision : {precision:.3f}")
print(f"  Recall    : {recall:.3f}")
print(f"  F1 Score  : {f1:.3f}")
print()
print(f"  Batches reviewed       : {len(batch_records)}")
print(f"  Batches flagged        : {len(flagged_batches)}")
print(f"  Clean batches (passed) : {len(batch_records) - len(flagged_batches)}")
print(f"  Review reduction       : {(len(batch_records)-len(flagged_batches))/len(batch_records)*100:.1f}% of batches need no manual exception review")

---
### Section 9 — Save outputs

In [ ]:
# Save all outputs
df_detection = pd.DataFrame(detection_log)
df_detection.to_csv("batch_detection_results.csv", index=False)

df_reviews = pd.DataFrame(review_summaries)
df_reviews.to_csv("batch_review_summaries.csv", index=False)

print("Files saved:")
print("  synthetic_batch_records.json   — 200 synthetic batch records")
print("  batch_detection_results.csv    — rule-based detection results")
print("  batch_review_summaries.csv     — AI-generated QA review summaries")
print()
print("SUMMARY")
print("=" * 40)
print(f"  Total batches          : {len(batch_records)}")
print(f"  Flagged for review     : {len(flagged_batches)}")
print(f"  Detection precision    : {precision:.3f}")
print(f"  Detection recall       : {recall:.3f}")
print(f"  Review summaries built : {len(review_summaries)}")
print()
print("NOTE: All data is synthetic. Production deployment requires UCB's")
print("actual MES batch record data under GxP-controlled access.")